In [7]:
import torch
from torch import nn
from torch.nn import functional as F
from d2l import torch as d2l

# Main Branch:
# X
# ↓ 1×1 Conv
# ↓ BatchNorm
# ↓ ReLU
# ↓ 3×3 Grouped Conv
# ↓ BatchNorm
# ↓ ReLU
# ↓ 1×1 Conv
# ↓ BatchNorm
# ├──────────────┐
# │              ▼
# └ Shortcut → Addition → ReLU

In [8]:
# ResNeXt Block

class ResNeXtBlock(nn.Module):
    def __init__(
        self,
        num_channels: int,
        groups: int,
        bot_mul: float,
        use_1x1conv: bool = False,
        strides: int = 1,
    ) -> None:
        super().__init__()

        # Intermediate bottleneck channel 수
        bot_channels = int(
            round(
                num_channels * bot_mul
            )
        )

        # D2L 구현에서 groups는 group 하나당 channel 수로 사용
        if (
            groups <= 0
            or bot_channels % groups != 0
        ):
            raise ValueError(
                "groups must be positive and divide bot_channels."
            )
            
        # PyTorch Conv2d에 전달할 실제 group 수
        num_groups = (
            bot_channels // groups
        )
        
        
        # Channel mixing과 bottleneck 생성
        self.conv1 = nn.LazyConv2d(
            out_channels=bot_channels,
            kernel_size=1,
            stride=1,
        )
        
        # Grouped Convolution
        self.conv2 = nn.LazyConv2d(
            out_channels=bot_channels,
            kernel_size=3,
            stride=strides,
            padding=1,
            groups=num_groups
        )
        
        # Channel mixing과 Output channel 복원
        self.conv3 = nn.LazyConv2d(
            out_channels=num_channels,
            kernel_size=1,
            stride=1,
        )
        
        self.bn1 = nn.LazyBatchNorm2d()
        self.bn2 = nn.LazyBatchNorm2d()
        self.bn3 = nn.LazyBatchNorm2d()
        
        
        # Projection Shortcut
        self.conv4: nn.Module | None = (
            nn.LazyConv2d(
                out_channels=num_channels,
                kernel_size=1,
                stride=strides,
            )
            if use_1x1conv
            else None
        )

        self.bn4: nn.Module | None = (
            nn.LazyBatchNorm2d()
            if use_1x1conv
            else None
        )
        
        
    def forward(
        self,
        X: torch.Tensor,
    ) -> torch.Tensor:
        
        # Main Branch
        Y = F.relu(
            self.bn1(
                self.conv1(X)
            )
        )
        
        Y = F.relu(
            self.bn2(
                self.conv2(Y)
            )
        )
        
        Y = self.bn3(
            self.conv3(Y)
        )
        
        # Shortcut Branch
        shortcut = X

        if (
            self.conv4 is not None
            and self.bn4 is not None
        ):
            shortcut = self.bn4(
                self.conv4(
                    shortcut
                )
            )

        return F.relu(
            Y + shortcut
        )

In [9]:
# Identity Shortcut 동작 확인

X = torch.randn(
    4,
    32,
    96,
    96,
)

identity_block = ResNeXtBlock(
    num_channels=32,
    groups=16,
    bot_mul=1.0,
)

with torch.no_grad():
    Y_identity = identity_block(
        X
    )

print(
    "Input shape:",
    tuple(X.shape),
)

print(
    "Output shape:",
    tuple(Y_identity.shape),
)

print(
    "Bottleneck channels:",
    identity_block.conv2.out_channels,
)

print(
    "Actual group count:",
    identity_block.conv2.groups,
)

Input shape: (4, 32, 96, 96)
Output shape: (4, 32, 96, 96)
Bottleneck channels: 32
Actual group count: 2


In [10]:
# Projection Shortcut 동작 확인

projection_block = ResNeXtBlock(
    num_channels=64,
    groups=16,
    bot_mul=1.0,
    use_1x1conv=True,
    strides=2,
)

with torch.no_grad():
    Y_projection = projection_block(
        X
    )

print(
    "Input shape:",
    tuple(X.shape),
)

print(
    "Output shape:",
    tuple(Y_projection.shape),
)

print(
    "Bottleneck channels:",
    projection_block.conv2.out_channels,
)

print(
    "Actual group count:",
    projection_block.conv2.groups,
)

Input shape: (4, 32, 96, 96)
Output shape: (4, 64, 48, 48)
Bottleneck channels: 64
Actual group count: 4
